# 03 — Fine-tune the richer-class hold detector (run on Google Colab)

**BoulderVision retrain.** Fine-tunes our existing `best.pt` on YOUR corrected
Roboflow dataset with the richer classes (`hold, volume, downclimb, marker, tape`).
We start from `best.pt` (not from scratch) to keep its strong hold detection.

## Before you run
1. `Runtime` → `Change runtime type` → Hardware accelerator → **T4 GPU**.
2. Have your Roboflow **Private API key** ready (Settings → API Keys). You paste it
   into a hidden prompt below — it is never stored in the notebook.
3. Have the file **`models/best.pt`** from the repo handy — you'll upload it when asked.

Run the cells top to bottom (`Shift+Enter`). Two cells pause for you: the API-key
prompt and the `best.pt` upload.

## 1. Confirm a GPU is attached
If this shows no GPU, set Runtime → Change runtime type → T4 GPU first.

In [ ]:
!nvidia-smi

## 2. Install dependencies (in the Colab VM)

In [ ]:
!pip install -q ultralytics roboflow

## 3. Download YOUR corrected dataset
Paste your Roboflow API key into the hidden prompt and press Enter. The
workspace / project / version below are already set to your `boulder` project, v2.

In [ ]:
from getpass import getpass
from roboflow import Roboflow

rf = Roboflow(api_key=getpass("Roboflow API key: "))
project = rf.workspace("dawids-workspace-tcdum").project("boulder-7ikod")
dataset = project.version(2).download("yolov8")
print("\ndownloaded to:", dataset.location)

## 4. Upload our `best.pt`
A file picker appears — choose `models/best.pt` from the repo (~22 MB).

In [ ]:
from google.colab import files
up = files.upload()            # pick best.pt
BEST = list(up.keys())[0]
print("uploaded:", BEST)

## 5. Inspect the classes
Note the order — we need it locally to set `filter.volume_class` after install.

In [ ]:
import yaml, os

data_yaml = os.path.join(dataset.location, "data.yaml")
d = yaml.safe_load(open(data_yaml))
print("classes:", d["names"], "| nc:", d.get("nc", len(d["names"])))

## 6. Train (fine-tune from `best.pt`)
Ultralytics adapts the detection head from 2 → 5 classes automatically — the
backbone keeps the learned hold features. ~1–2 h on a T4; `patience=20` stops
early if validation plateaus. Drop `epochs` to 30 for a quick first pass.

In [ ]:
from ultralytics import YOLO

model = YOLO(BEST)            # start from our strong hold detector
model.train(
    data=data_yaml,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    project="bouldervision",
    name="holds_v2",
)

## 7. Evaluate
`mAP50` is the headline number (higher is better; ~0.6+ is usable).

In [ ]:
m = model.val()
print("mAP50:   ", round(float(m.box.map50), 3))
print("mAP50-95:", round(float(m.box.map), 3))

## 8. Download the trained weights
Saves `best.pt` to your computer. Then locally: move it to `models/holds.pt`,
set `models.hold_detector: holds.pt` in `config/settings.yaml`, and (Etap 6)
set `filter.volume_class` to the index of `volume` from cell 5.

In [ ]:
from google.colab import files
best_path = str(model.trainer.best)   # exact path from the run you just trained
print("downloading:", best_path)
files.download(best_path)